# H. Pylori Detection Using FCOS
This notebook implements an anchor-free object detection model (FCOS) for H. Pylori detection using PyTorch and Torchvision.
We focus on:
1. **Design**: Data pipeline reading pre-processed Pascal VOC labels and FCOS architecture.
2. **Implementation**: Train/Validation split, training loop, and evaluation using Mean Average Precision (mAP).



## 1. Imports & Setup


In [1]:
import os
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as patches

import torchvision
from torchvision.models.detection import fcos_resnet50_fpn
from torchvision.models.detection.fcos import FCOSClassificationHead
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.model_selection import train_test_split
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Enable TF32 for Ampere GPUs (A100/A200) — faster matmuls with negligible precision loss
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True



/home/p240236cs/b230552cs_Shaheen/hpylori_fcos/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Design: Dataset & Data Loader
The dataset reads microscopic images and pre-converted Pascal VOC format labels (`[xmin, ymin, xmax, ymax]`) from the `pascal` directory.



In [2]:
class HPyloriDataset(Dataset):
    def __init__(self, image_paths, pascal_dir, transforms=None):
        self.image_paths = image_paths
        self.pascal_dir = pascal_dir
        self.transforms = transforms
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img_name = os.path.basename(img_path)
        
        # Open image
        image = Image.open(img_path).convert("RGB")
        
        # Look for corresponding label file in the pascal directory
        label_path = os.path.join(self.pascal_dir, os.path.splitext(img_name)[0] + ".txt")
        
        boxes = []
        labels = []
        
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        c, xmin, ymin, xmax, ymax = map(float, parts)
                        boxes.append([xmin, ymin, xmax, ymax])
                        labels.append(1) # H. Pylori is class 1
        
        # Apply Albumentations BEFORE converting to PyTorch tensors
        if self.transforms is not None:
            image_np = np.array(image)
            transformed = self.transforms(image=image_np, bboxes=boxes, labels=labels)
            image = transformed['image']
            boxes = transformed['bboxes']
            labels = transformed['labels']
            
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([idx])
        
        # Fallback if no transforms were provided
        if not torch.is_tensor(image):
            image = torchvision.transforms.functional.to_tensor(image)
            
        return image, target


# Helper function to collate batches
def collate_fn(batch):
    return tuple(zip(*batch))



## 3. Data Split & Augmentation
We split the data into 80% Training and 20% Validation using `sklearn.model_selection.train_test_split`.
Aggressive augmentation for training — including rotations, color/stain jitter, blur, and noise — to combat overfitting on the small dataset.



In [ ]:
images_dir = "data_2006/patches/images"
pascal_dir = "data_2006/patches/pascal"
test_images_dir = "data_2006/test_patches/images"
test_pascal_dir = "data_2006/test_patches/pascal"

all_image_paths = glob.glob(os.path.join(images_dir, "*.png"))

# Augmentation pipeline — aggressive for microscopy with H&E
train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=45, p=0.4,
                       border_mode=0),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.GaussNoise(p=1.0),
    ], p=0.2),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=15, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.CLAHE(clip_limit=2.0, p=0.15),
    # FCOS expects images in [0, 1] range — GeneralizedRCNNTransform handles ImageNet normalization internally
    A.Normalize(mean=(0,0,0), std=(1,1,1), max_pixel_value=255.0), 
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'],
                             min_area=64, min_visibility=0.3))

val_transforms = A.Compose([
    A.Normalize(mean=(0,0,0), std=(1,1,1), max_pixel_value=255.0),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

# Split training data (80% train, 20% val)
train_paths, val_paths = train_test_split(all_image_paths, test_size=0.2, random_state=42)

# Add dedicated test patches to validation set
test_patch_paths = glob.glob(os.path.join(test_images_dir, "*.png"))
print(f"Test patches found: {len(test_patch_paths)}")

# Combine val split + test patches for evaluation
all_val_paths = val_paths + test_patch_paths

# Training uses only the train split; validation includes both val split + test set
train_dataset = HPyloriDataset(train_paths, pascal_dir, transforms=train_transforms)

# For val, we need a dataset that can look up labels in both pascal dirs
# Since test patches have their own pascal dir, we create two datasets and concatenate
val_dataset_split = HPyloriDataset(val_paths, pascal_dir, transforms=val_transforms)
val_dataset_test = HPyloriDataset(test_patch_paths, test_pascal_dir, transforms=val_transforms)
val_dataset = torch.utils.data.ConcatDataset([val_dataset_split, val_dataset_test])

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)} (split: {len(val_dataset_split)} + test: {len(val_dataset_test)})")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)



Test patches found: 597
Training images: 19775
Validation images: 5541 (split: 4944 + test: 597)


/home/p240236cs/b230552cs_Shaheen/hpylori_fcos/venv/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/home/p240236cs/b230552cs_Shaheen/hpylori_fcos/venv/lib/python3.12/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


## 4. Design & Implementation: FCOS Architecture
We load the pre-trained FCOS model and replace the classification head for 2 classes (Background + H. Pylori).
Labels use class 1 for H. Pylori; torchvision FCOS expects labels in [1, num_classes].



In [4]:
def get_model(num_classes):
    # Load pre-trained FCOS model
    model = fcos_resnet50_fpn(weights="DEFAULT")
    
    # Replace the classification head for our number of classes
    in_channels = model.head.classification_head.conv[0].in_channels
    num_anchors = model.head.classification_head.num_anchors
    
    model.head.classification_head = FCOSClassificationHead(in_channels, num_anchors, num_classes)
    
    return model

# num_classes = 2 (Background + H. Pylori), labels use class 1
model = get_model(num_classes=2)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
print(f"Model loaded and moved to {device}")



Model loaded and moved to cuda


## 5. Implementation: Training & Validation Loop
Key training improvements:
1. **Backbone freezing**: Freeze ResNet-50 backbone for first 5 epochs so detection head learns meaningful features, then unfreeze with lower LR.
2. **Cosine annealing LR**: Smooth decay instead of aggressive step drops.
3. **Early stopping**: Save best model based on validation mAP, stop if no improvement for 10 epochs.
4. **Mixed precision**: FP16 training for 2x speedup on T4.



In [5]:
# Hyperparameters
num_epochs = 60
freeze_epochs = 5  # Number of epochs to freeze backbone
patience = 15      # Early stopping patience (more data = slower convergence)

# --- Phase 1: Freeze backbone, train only the detection head ---
for param in model.backbone.parameters():
    param.requires_grad = False

# Only optimize head parameters initially
head_params = [p for p in model.head.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(head_params, lr=0.01, momentum=0.9, weight_decay=0.0005)

metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
scaler = torch.cuda.amp.GradScaler()  # Mixed precision

best_map = 0.0
no_improve_count = 0

def train_one_epoch(model, optimizer, data_loader, device, epoch, scaler):
    model.train()
    total_loss = 0
    for i, (images, targets) in enumerate(data_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        
        # Mixed precision forward pass
        with torch.cuda.amp.autocast():
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
        
        # Mixed precision backward pass
        scaler.scale(losses).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += losses.item()
        
        if (i+1) % 20 == 0:
            print(f"Epoch [{epoch+1}] Batch [{i+1}/{len(data_loader)}] Loss: {losses.item():.4f}")
            
    avg_loss = total_loss/len(data_loader)
    print(f"Epoch [{epoch+1}] Average Training Loss: {avg_loss:.4f}")
    return avg_loss

@torch.no_grad()
def evaluate(model, data_loader, device, metric):
    model.eval()
    metric.reset()
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
        
        cpu_targets = [{k: v.cpu() for k, v in t.items()} for t in targets]
        cpu_outputs = [{k: v.cpu() for k, v in o.items()} for o in outputs]
        
        metric.update(cpu_outputs, cpu_targets)
        
    mAP_results = metric.compute()
    print(f"Validation mAP@0.5:0.95: {mAP_results['map']:.4f}")
    print(f"Validation mAP@0.5: {mAP_results['map_50']:.4f}")
    print(f"Validation mAP@0.75: {mAP_results['map_75']:.4f}")
    return mAP_results['map'].item(), mAP_results['map_50'].item()

print("=" * 60)
print("PHASE 1: Training detection head (backbone frozen)")
print("=" * 60)

for epoch in range(freeze_epochs):
    train_one_epoch(model, optimizer, train_loader, device, epoch, scaler)
    current_map, current_map50 = evaluate(model, val_loader, device, metric)
    if current_map > best_map:
        best_map = current_map
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  -> New best model saved! mAP={best_map:.4f}")

# --- Phase 2: Unfreeze backbone, fine-tune everything with lower LR ---
print()
print("=" * 60)
print("PHASE 2: Fine-tuning entire model (backbone unfrozen)")
print("=" * 60)

for param in model.backbone.parameters():
    param.requires_grad = True

# Differential learning rates: backbone gets 10x lower LR than head
optimizer = torch.optim.SGD([
    {'params': model.backbone.parameters(), 'lr': 0.0005},
    {'params': model.head.parameters(), 'lr': 0.005},
], momentum=0.9, weight_decay=0.0005)

remaining_epochs = num_epochs - freeze_epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining_epochs, eta_min=1e-6)

for epoch in range(freeze_epochs, num_epochs):
    train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch, scaler)
    scheduler.step()
    current_map, current_map50 = evaluate(model, val_loader, device, metric)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"  LR (backbone): {current_lr:.6f}, LR (head): {optimizer.param_groups[1]['lr']:.6f}")
    
    if current_map > best_map:
        best_map = current_map
        no_improve_count = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  -> New best model saved! mAP={best_map:.4f}")
    else:
        no_improve_count += 1
        print(f"  -> No improvement for {no_improve_count} epoch(s)")
        
    if no_improve_count >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

print()
print("=" * 60)
print(f"Training complete! Best mAP@0.5:0.95 = {best_map:.4f}")
print("Best model saved to best_model.pth")
print("=" * 60)



PHASE 1: Training detection head (backbone frozen)


/tmp/ipykernel_1908739/2716070339.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # Mixed precision
/tmp/ipykernel_1908739/2716070339.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [1] Batch [20/618] Loss: 1.7677
Epoch [1] Batch [40/618] Loss: 1.6434
Epoch [1] Batch [60/618] Loss: 1.6134
Epoch [1] Batch [80/618] Loss: 2.6386
Epoch [1] Batch [100/618] Loss: 1.6169
Epoch [1] Batch [120/618] Loss: 1.5770
Epoch [1] Batch [140/618] Loss: 1.4968
Epoch [1] Batch [160/618] Loss: 1.5304
Epoch [1] Batch [180/618] Loss: 1.5127
Epoch [1] Batch [200/618] Loss: 1.4886
Epoch [1] Batch [220/618] Loss: 1.6111
Epoch [1] Batch [240/618] Loss: 1.4753
Epoch [1] Batch [260/618] Loss: 1.5072
Epoch [1] Batch [280/618] Loss: 1.4653
Epoch [1] Batch [300/618] Loss: 1.4786
Epoch [1] Batch [320/618] Loss: 1.5947
Epoch [1] Batch [340/618] Loss: 1.5227
Epoch [1] Batch [360/618] Loss: 1.4711
Epoch [1] Batch [380/618] Loss: 1.4520
Epoch [1] Batch [400/618] Loss: 1.5658
Epoch [1] Batch [420/618] Loss: 1.4228
Epoch [1] Batch [440/618] Loss: 1.4679
Epoch [1] Batch [460/618] Loss: 1.4214
Epoch [1] Batch [480/618] Loss: 1.4585
Epoch [1] Batch [500/618] Loss: 1.4820
Epoch [1] Batch [520/618] Los

/tmp/ipykernel_1908739/2716070339.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Validation mAP@0.5:0.95: 0.0770
Validation mAP@0.5: 0.2550
Validation mAP@0.75: 0.0213
  -> New best model saved! mAP=0.0770
Epoch [2] Batch [20/618] Loss: 1.4008
Epoch [2] Batch [40/618] Loss: 1.4483
Epoch [2] Batch [60/618] Loss: 1.3828
Epoch [2] Batch [80/618] Loss: 1.4609
Epoch [2] Batch [100/618] Loss: 1.4354
Epoch [2] Batch [120/618] Loss: 1.4360
Epoch [2] Batch [140/618] Loss: 1.3965
Epoch [2] Batch [160/618] Loss: 1.4184
Epoch [2] Batch [180/618] Loss: 1.4782
Epoch [2] Batch [200/618] Loss: 1.4108
Epoch [2] Batch [220/618] Loss: 1.3762
Epoch [2] Batch [240/618] Loss: 1.4805
Epoch [2] Batch [260/618] Loss: 1.5177
Epoch [2] Batch [280/618] Loss: 1.4470
Epoch [2] Batch [300/618] Loss: 1.4075
Epoch [2] Batch [320/618] Loss: 1.5280
Epoch [2] Batch [340/618] Loss: 1.4157
Epoch [2] Batch [360/618] Loss: 1.3879
Epoch [2] Batch [380/618] Loss: 1.4209
Epoch [2] Batch [400/618] Loss: 1.5075
Epoch [2] Batch [420/618] Loss: 1.4907
Epoch [2] Batch [440/618] Loss: 1.4309
Epoch [2] Batch [460/

## 6. Final Evaluation with Best Model
Load the best checkpoint and run a final evaluation to report the definitive metrics.



In [8]:
# Warm restart from best checkpoint
model.load_state_dict(torch.load("best_model.pth", weights_only=True))
model.to(device)

optimizer = torch.optim.SGD([
    {'params': model.backbone.parameters(), 'lr': 0.0002},
    {'params': model.head.parameters(), 'lr': 0.002},
], momentum=0.9, weight_decay=0.0005)

restart_epochs = 30
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=restart_epochs, eta_min=1e-6)

best_map = 0.1829
best_map50 = 0.5265
no_improve = 0

for epoch in range(restart_epochs):
    train_loss = train_one_epoch(model, optimizer, train_loader, device, 60 + epoch, scaler)
    scheduler.step()
    current_map, current_map50 = evaluate(model, val_loader, device, metric)
    
    lr_bb = optimizer.param_groups[0]['lr']
    lr_hd = optimizer.param_groups[1]['lr']
    print(f"  LR: backbone={lr_bb:.6f}, head={lr_hd:.6f}")
    
    if current_map > best_map:
        best_map = current_map
        best_map50 = current_map50
        no_improve = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  -> New best! mAP@0.5:0.95={best_map:.4f}, mAP@0.5={best_map50:.4f}")
    else:
        no_improve += 1
        print(f"  -> No improvement for {no_improve} epoch(s) (best={best_map:.4f})")
        if no_improve >= 10:
            print(f"Early stopping at epoch {60 + epoch + 1}")
            break

print(f"\nDone! Best mAP@0.5:0.95={best_map:.4f}, mAP@0.5={best_map50:.4f}")


/tmp/ipykernel_1908739/2716070339.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [61] Batch [20/618] Loss: 1.1887
Epoch [61] Batch [40/618] Loss: 1.2152
Epoch [61] Batch [60/618] Loss: 1.2705
Epoch [61] Batch [80/618] Loss: 1.2523
Epoch [61] Batch [100/618] Loss: 1.2768
Epoch [61] Batch [120/618] Loss: 1.2230
Epoch [61] Batch [140/618] Loss: 1.2971
Epoch [61] Batch [160/618] Loss: 1.2540
Epoch [61] Batch [180/618] Loss: 1.2425
Epoch [61] Batch [200/618] Loss: 1.2967
Epoch [61] Batch [220/618] Loss: 1.2239
Epoch [61] Batch [240/618] Loss: 1.1839
Epoch [61] Batch [260/618] Loss: 1.2783
Epoch [61] Batch [280/618] Loss: 1.2683
Epoch [61] Batch [300/618] Loss: 1.1864
Epoch [61] Batch [320/618] Loss: 1.2939
Epoch [61] Batch [340/618] Loss: 1.2228
Epoch [61] Batch [360/618] Loss: 1.2200
Epoch [61] Batch [380/618] Loss: 1.2165
Epoch [61] Batch [400/618] Loss: 1.3385
Epoch [61] Batch [420/618] Loss: 1.2161
Epoch [61] Batch [440/618] Loss: 1.3176
Epoch [61] Batch [460/618] Loss: 1.2304
Epoch [61] Batch [480/618] Loss: 1.2486
Epoch [61] Batch [500/618] Loss: 1.2604
Epoc

/tmp/ipykernel_1908739/2716070339.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Validation mAP@0.5:0.95: 0.1819
Validation mAP@0.5: 0.5232
Validation mAP@0.75: 0.0570
  LR: backbone=0.000199, head=0.001995
  -> No improvement for 1 epoch(s) (best=0.1829)
Epoch [62] Batch [20/618] Loss: 1.2372
Epoch [62] Batch [40/618] Loss: 1.2141
Epoch [62] Batch [60/618] Loss: 1.2286
Epoch [62] Batch [80/618] Loss: 1.2109
Epoch [62] Batch [100/618] Loss: 1.3258
Epoch [62] Batch [120/618] Loss: 1.2379
Epoch [62] Batch [140/618] Loss: 1.2738
Epoch [62] Batch [160/618] Loss: 1.2765
Epoch [62] Batch [180/618] Loss: 1.2204
Epoch [62] Batch [200/618] Loss: 1.3012
Epoch [62] Batch [220/618] Loss: 1.2739
Epoch [62] Batch [240/618] Loss: 1.3382
Epoch [62] Batch [260/618] Loss: 1.1991
Epoch [62] Batch [280/618] Loss: 1.1997
Epoch [62] Batch [300/618] Loss: 1.2308
Epoch [62] Batch [320/618] Loss: 1.2559
Epoch [62] Batch [340/618] Loss: 1.3000
Epoch [62] Batch [360/618] Loss: 1.2159
Epoch [62] Batch [380/618] Loss: 1.1954
Epoch [62] Batch [400/618] Loss: 1.2867
Epoch [62] Batch [420/618] Lo

In [9]:
# Load best model and evaluate
model.load_state_dict(torch.load("best_model.pth"))
model.to(device)

print("Final evaluation with best checkpoint:")
final_map, final_map50 = evaluate(model, val_loader, device, metric)
print(f"\nFinal mAP@0.5:0.95 = {final_map:.4f}")
print(f"Final mAP@0.5     = {final_map50:.4f}")



Final evaluation with best checkpoint:


/tmp/ipykernel_1908739/2716070339.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Validation mAP@0.5:0.95: 0.1829
Validation mAP@0.5: 0.5265
Validation mAP@0.75: 0.0622

Final mAP@0.5:0.95 = 0.1829
Final mAP@0.5     = 0.5265
